> **Versión histórica v2** de `08_etl_transformaciones_kpis`. **Sustituida por v4 (versión final).**
> Qué trae: T1–T5 leyendo los CSV originales, 6 KPIs con histórico, secciones 2-3 (tipo de problema y feature engineering) y serialización con joblib.
> Limitaciones que corrige v3/v4: no vacía las tablas T1–T5 al relanzar (duplica filas), no cuadra T1 contra la estrella y exige haber ejecutado `modelo_dwh_mysql.sql` a mano.
> Archivo equivalente: `v2` y `v2_1` eran idénticos (v2_1 solo añadía 28 celdas vacías al final).

# 08 — ETL: Transformaciones, Feature Engineering y KPIs
### TFM · Análisis multidimensional del estrés estudiantil · Entrega 2

---
## Estructura de este notebook

| Sección | Contenido | Alineado con |
|---|---|---|
| 1. Configuración | Conexión MySQL, funciones de armonización | — |
| 2. Tipo de problema | Clasificación de cada dataset (binaria/multiclase) | Días 5-6 |
| 3. Feature Engineering | Encoding, variables derivadas, justificación | Día 8 |
| T1-T5 | Cinco transformaciones ETL → tablas DWH | Entrega 2 |
| KPIs | Seis indicadores con histórico de ejecuciones | Entrega 2 |
| Serialización | Guardado de pipeline con joblib | Día 11 |

**Requisitos previos:**
- XAMPP con MySQL en verde
- `modelo_datos_mysql_v2.sql` y `esquema_estrella_powerbi.sql` importados en phpMyAdmin
- `modelo_dwh_mysql.sql` importado (crea las tablas DWH vacías)
- Notebook `07_carga_datos_mysql.ipynb` ejecutado (datos cargados)

## Paso 0 — Configuración y conexión

In [ ]:
import os, json
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime
from sqlalchemy import create_engine, text

# ── Rutas ──────────────────────────────────────────────────────────────────
BASE = r"C:\Users\nicop\OneDrive\Documentos\Maestria\INESDI & UNIE\BA & IA\TFM\Propuesta"
ALMC = r"C:\Users\nicop\OneDrive\Documentos\Maestria\INESDI & UNIE\BA & IA\TFM\Propuesta\Entrega\Objetivo2"
FIG  = os.path.join(ALMC, "figuras_etl")
os.makedirs(FIG, exist_ok=True)

# ── Conexión MySQL (XAMPP/MariaDB) ─────────────────────────────────────────
DB_USER, DB_PASS = "root", ""
DB_HOST, DB_NAME = "localhost", "tfm_estres_estudiantil_2"
engine = create_engine(f"mysql+pymysql://{DB_USER}:{DB_PASS}@{DB_HOST}/{DB_NAME}?charset=utf8mb4")

EJECUCION = datetime.now()
print("Conectado a", DB_NAME)
print("Ejecución:", EJECUCION.strftime("%Y-%m-%d %H:%M:%S"))

---
## Sección 2 — Tipo de problema por dataset

Antes de transformar y modelar, es fundamental clasificar cada problema de ML: el tipo
condiciona el algoritmo, la métrica de evaluación y el tratamiento del desequilibrio de clases.
*(Referencia: Días 5-6, Daniel Toral)*

In [ ]:
# Clasificación del tipo de problema por dataset (Días 5-6, Daniel Toral)
tipos_problema = {
    "B1_ACADSTRESS":    {"tipo": "Clasificación multiclase",
                         "target": "indice_estres (1-5)",
                         "clases": 5, "balance": "desbalanceado (clases 1-2 < 11%)",
                         "metrica": "F1-macro (evita sesgo por clase mayoritaria)"},
    "B1_ANXIETY":       {"tipo": "Clasificación binaria (umbral GAD-7)",
                         "target": "nivel_estres (Bajo/Alto)",
                         "clases": 2, "balance": "desbalanceado",
                         "metrica": "AUC-ROC (robusto ante desbalance)"},
    "B1_MHSURVEY":      {"tipo": "Clasificación ternaria (Bajo/Mod/Alto)",
                         "target": "depresion (1-5 → Bajo/Mod/Alto)",
                         "clases": 3, "balance": "moderado (45/31/24%)",
                         "metrica": "F1-macro"},
    "B2_SMMH":          {"tipo": "Clasificación binaria",
                         "target": "se_siente_deprimido (1-5 → Bajo/Alto)",
                         "clases": 2, "balance": "casi balanceado (54/46%)",
                         "metrica": "AUC-ROC + F1"},
    "B3_DREADDIT":      {"tipo": "Clasificación binaria (NLP)",
                         "target": "label (0=No estres / 1=Estres)",
                         "clases": 2, "balance": "casi balanceado (1350/1488)",
                         "metrica": "Accuracy + F1 + AUC"},
    "B3_DEPSTUDENT":    {"tipo": "Clasificación binaria",
                         "target": "depresion (Yes/No)",
                         "clases": 2, "balance": "balanceado (252/250)",
                         "metrica": "Accuracy + AUC"},
    "B4_DASS":          {"tipo": "Clasificación multiclase por severidad clínica",
                         "target": "nivel_estres (Bajo/Moderado/Alto, baremos DASS)",
                         "clases": 3, "balance": "casi balanceado (30/35/35%)",
                         "metrica": "F1-macro + AUC multiclase (OvR)"},
    "B4_MENTALHEALTH":  {"tipo": "Clasificación binaria",
                         "target": "tratamiento (Yes/No)",
                         "clases": 2, "balance": "casi balanceado (49/51%)",
                         "metrica": "F1 + AUC"},
}

print(f"{'Dataset':<20} {'Tipo':<35} {'Métrica recomendada'}")
print("-" * 85)
for ds, info in tipos_problema.items():
    print(f"{ds:<20} {info['tipo']:<35} {info['metrica']}")

print("""
─────────────────────────────────────────────────────────────
Por qué NO usamos solo Accuracy (Día 5, Daniel Toral):
  • Accuracy es engañoso cuando las clases están desbalanceadas:
    en Anxiety solo el 17% alcanza GAD-7>=10: predecir siempre "no alto"
    da 83% de Accuracy con F1=0 (no detecta ningún caso).
  • AUC-ROC evalúa la capacidad discriminante del modelo
    independientemente del umbral → válido con desequilibrio.
  • F1-score balancea precisión y recall → útil cuando los
    falsos negativos tienen coste clínico alto (p. ej. no
    detectar depresión real).
─────────────────────────────────────────────────────────────""")

**Opinión.** La mayoría de los datasets presentan clasificación binaria, que es el problema de ML más directo. Los casos multiclase (Academic Stress, DASS) requerirán estrategias One-vs-Rest para AUC y métricas macro para tratar equitativamente las clases minoritarias. Esta clasificación previa evita el error típico de aplicar métricas inadecuadas — documentado en clase como uno de los fallos más frecuentes.

---
## Sección 3 — Feature Engineering

Transformaciones necesarias para que los modelos del Objetivo 2 puedan procesar cada dataset.
Se aplican tres técnicas del Día 8 (Álvaro López): Label Encoding (variables ordinales),
One-Hot Encoding (variables nominales sin orden) y variables derivadas/interacción.

In [ ]:
# ── Feature Engineering: resumen de decisiones de codificación por dataset ──
# (Día 8, Álvaro López — Label Encoding / One-Hot / Variables derivadas)

fe_decisions = {
    "B1_ACADSTRESS": {
        "label_enc":  ["etapa_academica (high school=0, undergraduate=1, post-graduate=2)"],
        "onehot_enc": ["entorno_estudio", "estrategia_afrontamiento"],
        "derivadas":  ["presion_total = presion_pares + presion_hogar + competencia_academica"],
        "excluir":    ["ts_respuesta (temporal, no predictiva)", "habitos_nocivos (texto libre)"],
    },
    "B1_ANXIETY": {
        "label_enc":  ["ocupacion_est (Student=1, otros=0)", "nivel_educativo (ordinal)"],
        "onehot_enc": [],
        "derivadas":  ["gad_severo = 1 si GAD_T>=10", "ansiedad_social_alta = 1 si SPIN_T>=20"],
        "excluir":    ["gade (subjetivo)", "pais_residencia (alta cardinalidad)"],
    },
    "B1_MHSURVEY": {
        "label_enc":  ["cgpa_rango (rangos a ordinal: 2.0-2.5=0 ... 3.5-4.0=3)",
                       "average_sleep (rangos a horas: <5=0 ... >9=4)"],
        "onehot_enc": ["residential_status"],
        "derivadas":  ["carga_presion = carga_academica * presion_academica (interaccion)",
                       "bienestar = satisfaccion_estudios - aislamiento"],
        "excluir":    ["universidad", "carrera (alta cardinalidad)"],
    },
    "B2_SMMH": {
        "label_enc":  [],
        "onehot_enc": ["plataformas (split multi-seleccion → dummies por plataforma)"],
        "derivadas":  ["uso_ansioso = (inquietud_sin_redes + distraccion_facil + preocupacion) / 3",
                       "impacto_social = (comparacion_social + busqueda_validacion) / 2"],
        "excluir":    ["ts_respuesta", "estado_relacion"],
    },
    "B3_DREADDIT": {
        "label_enc":  ["subreddit (target encoding por clase estres media del subreddit)"],
        "onehot_enc": [],
        "derivadas":  ["ansiedad_vs_positivo = lex_liwc_anx / (lex_liwc_posemo + 0.001)",
                       "n_words (ya calculado en EDA)"],
        "excluir":    ["texto (se usa vía TF-IDF en Objetivo 2)", "post_id_origen", "ts_publicacion"],
    },
    "B3_DEPSTUDENT": {
        "label_enc":  ["duracion_sueno (Less than 5=0 ... More than 9=4)",
                       "habitos_alimentarios (Unhealthy=0, Moderate=1, Healthy=2)"],
        "onehot_enc": [],
        "derivadas":  ["riesgo_alto = 1 si presion_academica>=4 AND estres_financiero>=4"],
        "excluir":    [],
    },
    "B4_DASS": {
        "label_enc":  ["genero (codebook: 1/2/3)", "educacion (1-4 ya ordinal)"],
        "onehot_enc": [],
        "derivadas":  ["malestar_total = score_depresion + score_ansiedad + score_estres",
                       "perfil_severo = 1 si score_estres>=40 AND score_depresion>=38"],
        "excluir":    ["pais (alta cardinalidad)", "carrera", "seg_intro/test/encuesta"],
    },
    "B4_MENTALHEALTH": {
        "label_enc":  ["estres_creciente (No=0, Maybe=1, Yes=2)",
                       "cambio_habitos", "historial_salud_mental", "cambios_humor"],
        "onehot_enc": [],
        "derivadas":  ["vulnerabilidad = antecedentes_familiares + historial_salud_mental (sum)"],
        "excluir":    ["pais (alta cardinalidad)", "SocialWeakness.1 (duplicado)"],
    },
}

print("DECISIONES DE FEATURE ENGINEERING POR DATASET (Día 8, Álvaro López)")
print("=" * 70)
for ds, fe in fe_decisions.items():
    print(f"\n▶ {ds}")
    if fe["label_enc"]:
        print(f"  Label Encoding (ordinal): {'; '.join(fe['label_enc'][:2])}")
    if fe["onehot_enc"]:
        print(f"  One-Hot (nominal):        {'; '.join(fe['onehot_enc'])}")
    if fe["derivadas"]:
        print(f"  Variables derivadas:      {'; '.join(fe['derivadas'][:2])}")
    if fe["excluir"]:
        print(f"  Excluir:                  {'; '.join(fe['excluir'][:2])}")

**Opinión.** La decisión de codificación es crítica: aplicar Label Encoding a variables nominales (sin orden) introduce relaciones falsas que el modelo aprenderá como ruido. Por ejemplo, codificar plataformas de redes sociales (Instagram=1, YouTube=2, TikTok=3) haría que el modelo infiera que TikTok es "más" que Instagram en algún sentido. Las variables derivadas de interacción (`carga_presion`, `uso_ansioso`) son el aporte de Feature Engineering más relevante: condensan señales relacionadas en una sola variable. Ojo: `uso_ansioso` (r≈0.56 con el target) no supera a su mejor componente, `preocupacion` (r≈0.59); su ventaja es la estabilidad al promediar tres señales, no una correlación mayor. La implementación completa de estas técnicas sobre los ocho datasets está en el notebook 09.

In [ ]:
# Ejemplo ejecutable: Feature Engineering en B2_SMMH (el más rico del Bloque 2)
d = pd.read_csv(os.path.join(BASE, "Bloque 2", "2 Social Media and Mental Health", "smmh.csv"))
c = list(d.columns)

# 1. Dummies por plataforma (One-Hot sobre multi-selección)
plataformas = ["Facebook","Instagram","YouTube","TikTok","Snapchat","Twitter","Pinterest","Reddit"]
for p in plataformas:
    d[f"usa_{p.lower()}"] = d[c[7]].str.contains(p, na=False).astype(int)

# 2. Variables derivadas de interacción
d["uso_ansioso"]    = (pd.to_numeric(d[c[11]], errors="coerce").fillna(0) +
                       pd.to_numeric(d[c[12]], errors="coerce").fillna(0) +
                       pd.to_numeric(d[c[13]], errors="coerce").fillna(0)) / 3
d["impacto_social"] = (pd.to_numeric(d[c[15]], errors="coerce").fillna(0) +
                       pd.to_numeric(d[c[17]], errors="coerce").fillna(0)) / 2
d["target"]         = pd.to_numeric(d[c[18]], errors="coerce")

print(f"SMMH tras Feature Engineering: {d.shape[0]} filas x {d.shape[1]} columnas")
print("Nuevas variables creadas:", [col for col in d.columns if col.startswith("usa_") or col in ["uso_ansioso","impacto_social"]])
print(f"\nCorrelación 'uso_ansioso' -> target: {d['uso_ansioso'].corr(d['target']):.3f}")
print(f"Correlación 'impacto_social' -> target: {d['impacto_social'].corr(d['target']):.3f}")

---
## Transformaciones ETL → Datawarehouse

Las siguientes cinco transformaciones generan las tablas `dwh_*` del datawarehouse. **Requisito:** haber ejecutado `modelo_dwh_mysql.sql` en phpMyAdmin antes de correr estas celdas.

In [ ]:
# ── Funciones de armonización (mismas que notebook 06/07) ──────────────────
def map_genero(v):
    if pd.isna(v): return "NR"
    s = str(v).strip().lower()
    if s in ("male","m","hombre","1","1.0"):  return "M"
    if s in ("female","f","mujer","2","2.0"): return "F"
    if s in ("","nan","prefer not to say"):   return "NR"
    return "O"

def map_rango_edad(v):
    try: e = float(v)
    except: return "NR"
    if pd.isna(e) or e<13 or e>90: return "NR"
    if e<18: return "<18"
    if e<=25: return "18-25"
    if e<=35: return "26-35"
    if e<=50: return "36-50"
    return ">50"

def likert(v):
    try: x=float(v)
    except: return "No aplica"
    return "Bajo" if x<=2 else "Moderado" if x<=3 else "Alto"

def gad(v):
    try: x=float(v)
    except: return "No aplica"
    return "Bajo" if x<=4 else "Moderado" if x<=9 else "Alto"

def dass(v): return "Bajo" if v<=28 else "Moderado" if v<=39 else "Alto"

GEN_MAP = {"M":"Masculino","F":"Femenino","O":"Otro","NR":"No reportado"}
ID_GEN  = {"Masculino":1,"Femenino":2,"Otro":3,"No reportado":0}
ID_RNG  = {"No reportado":0,"NR":0,"<18":1,"18-25":2,"26-35":3,"36-50":4,">50":5}
ID_EDU  = {"No reportado":0,"Secundaria o menos":1,"Grado universitario":2,"Posgrado":3}
ID_EST  = {"No aplica":0,"Bajo":1,"Moderado":2,"Alto":3}
ID_DS   = {"B1_ACADSTRESS":1,"B1_ANXIETY":2,"B1_MHSURVEY":3,"B2_SMMH":6,
           "B3_DREADDIT_TR":8,"B3_DREADDIT_TE":9,"B3_DEPSTUDENT":10,"B4_DASS":11,"B4_MENTALHEALTH":12}
NR = lambda n: ["NR"]*n
EDUNR = lambda n: ["No reportado"]*n
print("Funciones listas.")

### T1 — Hechos unificados (integración de 9 fuentes reales)

`dwh_hechos_estres_unificado` — Une las 9 fuentes con señal real (uso_modelado='incluido') en una tabla de hechos con claves conformes. Excluye los 3 datasets sintéticos conforme a la política acordada.

In [ ]:
registros = []
def agregar(codigo, bloque, serie_gen, serie_edad, serie_edu, serie_estres):
    gen = [map_genero(x) for x in serie_gen]
    for (gcod, glbl), ed, edu, es in zip(
            [(g, GEN_MAP[g]) for g in gen], serie_edad, serie_edu, serie_estres):
        registros.append({"codigo_dataset":codigo,"bloque":bloque,
            "genero":glbl,"rango_edad":ed,"nivel_educativo":edu,"nivel_estres":es,
            "demografia_completa":(gcod!="NR" and ed!="NR")})

d=pd.read_csv(os.path.join(BASE,"Bloque 1","1 Student Academic Stress Real World Dataset","academic Stress level - maintainance 1.csv")); d.columns=[c.strip() for c in d.columns]
edu=d["Your Academic Stage"].map({"undergraduate":"Grado universitario","post-graduate":"Posgrado","high school":"Secundaria o menos"}).fillna("No reportado")
agregar("B1_ACADSTRESS","Factores internos",NR(len(d)),NR(len(d)),edu,d["Rate your academic stress index"].map(likert))

d=pd.read_csv(os.path.join(BASE,"Bloque 1","1 Student anxiety dataset","anxiety.csv"),encoding="latin1")
agregar("B1_ANXIETY","Factores internos",d["Gender"],d["Age"].map(map_rango_edad),EDUNR(len(d)),d["GAD_T"].map(gad))

d=pd.read_csv(os.path.join(BASE,"Bloque 1","1 Student Mental Health Survey","MentalHealthSurvey.csv")); d.columns=[c.strip() for c in d.columns]
edu=d["degree_level"].map({"Undergraduate":"Grado universitario","Postgraduate":"Posgrado"}).fillna("No reportado")
agregar("B1_MHSURVEY","Factores internos",d["gender"],d["age"].map(map_rango_edad),edu,d["depression"].map(likert))

d=pd.read_csv(os.path.join(BASE,"Bloque 2","2 Social Media and Mental Health","smmh.csv")); c=list(d.columns)
agregar("B2_SMMH","Habitos digitales",d[c[2]],d[c[1]].map(map_rango_edad),EDUNR(len(d)),pd.to_numeric(d[c[18]],errors="coerce").map(likert))

for part in ["train","test"]:
    d=pd.read_csv(os.path.join(BASE,"Bloque 3","3 Stress Analysis in Social Media Dataset",f"dreaddit-{part}.csv"))
    d=d[~d["text"].str.strip().isin(["#NAME?","#REF!","#VALUE!"])]
    agregar(f"B3_DREADDIT_{part[:2].upper()}","Percepcion emocional",NR(len(d)),NR(len(d)),EDUNR(len(d)),d["label"].map({0:"Bajo",1:"Alto"}))

d=pd.read_csv(os.path.join(BASE,"Bloque 3","3 Student_Mental_Health_Analysis GITHUB","Depression+Student+Dataset.csv"))
agregar("B3_DEPSTUDENT","Percepcion emocional",d["Gender"],d["Age"].map(map_rango_edad),EDUNR(len(d)),d["Depression"].map({"No":"Bajo","Yes":"Alto"}))

d=pd.read_csv(os.path.join(BASE,"Bloque 4","4 Depression Anxiety Stress Scales Responses","data.csv"),sep="\t",low_memory=False)
d=d[(d["age"]>=13)&(d["age"]<=90)].copy()
stri=[1,6,8,11,12,14,18,22,27,29,32,33,35,39]; d["strs"]=d[[f"Q{i}A" for i in stri]].sum(axis=1)
edu=d["education"].map({1:"Secundaria o menos",2:"Secundaria o menos",3:"Grado universitario",4:"Posgrado"}).fillna("No reportado")
agregar("B4_DASS","Validacion psicologica",d["gender"],d["age"].map(map_rango_edad),edu,d["strs"].map(dass))

d=pd.read_csv(os.path.join(BASE,"Bloque 4","4 Mental Health Dataset","Mental Health dataset1.csv"))
agregar("B4_MENTALHEALTH","Validacion psicologica",d["Gender"],NR(len(d)),EDUNR(len(d)),d["IncreasingStress"].map({"No":"Bajo","Maybe":"Moderado","Yes":"Alto"}).fillna("No aplica"))

T1=pd.DataFrame(registros)
T1["id_dataset"]=T1["codigo_dataset"].map(ID_DS)
T1["id_genero"]=T1["genero"].map(ID_GEN)
T1["id_rango_edad"]=T1["rango_edad"].map(ID_RNG)
T1["id_nivel_educativo"]=T1["nivel_educativo"].map(ID_EDU)
T1["id_nivel_estres"]=T1["nivel_estres"].map(ID_EST)
T1=T1[["id_dataset","codigo_dataset","bloque","id_genero","genero","id_rango_edad","rango_edad","id_nivel_educativo","nivel_educativo","id_nivel_estres","nivel_estres","demografia_completa"]]
T1.to_sql("dwh_hechos_estres_unificado",engine,if_exists="append",index=False,chunksize=5000)
print(f"T1: {len(T1):,} registros → dwh_hechos_estres_unificado")
print(T1["bloque"].value_counts().to_string())

**Opinión.** La unificación confirma que la Validación Psicológica domina en volumen (94%). Para los dashboards del Objetivo 3 esto es ventaja (base estadística robusta); para el modelado del Objetivo 2 se trabajará dataset por dataset, manteniendo el enfoque de análisis en paralelo.

### T2 — Agregado de prevalencia (agregación)

In [ ]:
T2=T1.groupby(["bloque","genero","rango_edad"]).agg(
    n_total=("nivel_estres","size"),
    n_estres_alto=("nivel_estres",lambda s:(s=="Alto").sum()),
    n_estres_moderado=("nivel_estres",lambda s:(s=="Moderado").sum()),
    n_estres_bajo=("nivel_estres",lambda s:(s=="Bajo").sum())).reset_index()
T2["pct_estres_alto"]=(T2["n_estres_alto"]/T2["n_total"]*100).round(2)
T2.to_sql("dwh_agg_prevalencia",engine,if_exists="append",index=False)
print(f"T2: {len(T2)} grupos → dwh_agg_prevalencia")
display(T2.sort_values("pct_estres_alto",ascending=False).head(8))

**Opinión.** La variabilidad por grupo demográfico es real: confirma que género y rango de edad añaden señal predictiva — justificando su inclusión como features en los modelos del Objetivo 2.

### T3 — Señales NLP de Dreaddit (resumen estadístico por grupo)

In [ ]:
tr=pd.read_csv(os.path.join(BASE,"Bloque 3","3 Stress Analysis in Social Media Dataset","dreaddit-train.csv"))
te=pd.read_csv(os.path.join(BASE,"Bloque 3","3 Stress Analysis in Social Media Dataset","dreaddit-test.csv"))
dr=pd.concat([tr,te],ignore_index=True)
dr=dr[~dr["text"].str.strip().isin(["#NAME?","#REF!","#VALUE!"])].copy()
dr["clase_estres"]=dr["label"].map({0:"No estres",1:"Estres"})
dr["n_words"]=dr["text"].str.split().str.len()
T3=dr.groupby(["subreddit","clase_estres"]).agg(
    n_posts=("text","size"),sentimiento_medio=("sentiment","mean"),
    liwc_ansiedad_medio=("lex_liwc_anx","mean"),liwc_emo_neg_medio=("lex_liwc_negemo","mean"),
    liwc_emo_pos_medio=("lex_liwc_posemo","mean"),longitud_media_palabras=("n_words","mean")
).round(4).reset_index()
T3.to_sql("dwh_nlp_senales",engine,if_exists="append",index=False)
print(f"T3: {len(T3)} combinaciones → dwh_nlp_senales")
display(T3.head(6))

**Opinión.** La señal LIWC es consistente en todos los subreddits: los posts con estrés siempre tienen mayor densidad de palabras de ansiedad. Esto avala la selección de features LIWC como predictores fuertes para el modelo NLP del Objetivo 2, antes de necesitar TF-IDF o embeddings.

### T4 — DASS por demografía con severidad clínica (pivote + clasificación por umbrales)

In [ ]:
d=pd.read_csv(os.path.join(BASE,"Bloque 4","4 Depression Anxiety Stress Scales Responses","data.csv"),sep="\t",low_memory=False)
d=d[(d["age"]>=18)&(d["age"]<=25)].copy()
depi=[3,5,10,13,16,17,21,24,26,31,34,37,38,42]; anxi=[2,4,7,9,15,19,20,23,25,28,30,36,40,41]; stri=[1,6,8,11,12,14,18,22,27,29,32,33,35,39]
d["dep"]=d[[f"Q{i}A" for i in depi]].sum(axis=1); d["anx"]=d[[f"Q{i}A" for i in anxi]].sum(axis=1); d["str"]=d[[f"Q{i}A" for i in stri]].sum(axis=1)
d["genero"]=d["gender"].map({1:"Masculino",2:"Femenino",3:"Otro"}).fillna("No reportado")
d["rango_edad"]="18-25"
d["nivel_educativo"]=d["education"].map({1:"Secundaria o menos",2:"Secundaria o menos",3:"Grado universitario",4:"Posgrado"}).fillna("No reportado")
T4=d.groupby(["genero","rango_edad","nivel_educativo"]).agg(
    n_respondentes=("dep","size"),depresion_media=("dep","mean"),ansiedad_media=("anx","mean"),
    estres_medio=("str","mean"),pct_estres_severo=("str",lambda s:(s>=40).mean()*100)
).round(2).reset_index()
T4.to_sql("dwh_dass_severidad",engine,if_exists="append",index=False)
print(f"T4: {len(T4)} grupos demográficos → dwh_dass_severidad")
display(T4.sort_values("estres_medio",ascending=False).head(6))

### T5 — Brecha de tratamiento (tabla de contingencia)

In [ ]:
mh=pd.read_csv(os.path.join(BASE,"Bloque 4","4 Mental Health Dataset","Mental Health dataset1.csv"))
mh=mh[mh["Occupation"]=="Student"].copy()
filas=[]
for dim,col in [("estres_creciente","IncreasingStress"),("antecedentes_familiares","FamilyHistory")]:
    for cat,sub in mh.groupby(col):
        filas.append({"dimension":dim,"categoria":str(cat),"n_total":len(sub),"n_con_tratamiento":int((sub["Treatment"]=="Yes").sum())})
T5=pd.DataFrame(filas); T5["pct_tratamiento"]=(T5["n_con_tratamiento"]/T5["n_total"]*100).round(2)
T5.to_sql("dwh_brecha_tratamiento",engine,if_exists="append",index=False)
print(f"T5: {len(T5)} filas → dwh_brecha_tratamiento")
display(T5)

---
## Cálculo de KPIs y almacenamiento con histórico

In [ ]:
kpis=[]
v=(T1["nivel_estres"]=="Alto").mean()*100
kpis.append((1,"KPI_ESTRES_ALTO",round(v,4),f"{(T1['nivel_estres']=='Alto').sum():,} de {len(T1):,}"))
amuj=d[d["genero"]=="Femenino"]["anx"].mean(); ahom=d[d["genero"]=="Masculino"]["anx"].mean()
kpis.append((2,"KPI_BRECHA_GENERO",round(amuj-ahom,4),f"Mujer {amuj:.1f} - Hombre {ahom:.1f}"))
dp=pd.read_csv(os.path.join(BASE,"Bloque 3","3 Student_Mental_Health_Analysis GITHUB","Depression+Student+Dataset.csv"))
ct=pd.crosstab(dp["Have you ever had suicidal thoughts ?"],dp["Depression"],normalize="index")*100
kpis.append((3,"KPI_RIESGO_CRITICO",round(ct.loc["Yes","Yes"],4),"% suicidas con depresion"))
kpis.append((4,"KPI_TASA_TRATAMIENTO",round((mh["Treatment"]=="Yes").mean()*100,4),f"{(mh['Treatment']=='Yes').sum():,} estudiantes"))
ae=dr[dr["clase_estres"]=="Estres"]["lex_liwc_anx"].mean(); an=dr[dr["clase_estres"]=="No estres"]["lex_liwc_anx"].mean()
kpis.append((5,"KPI_DISCRIMINANTE_LIWC",round(ae/an,4),f"estres {ae:.2f} / no-estres {an:.2f}"))
kpis.append((6,"KPI_COBERTURA_DEMO",round(T1["demografia_completa"].mean()*100,4),"% con genero+edad reportados"))
res=pd.DataFrame(kpis,columns=["id_kpi","codigo","valor","detalle"]); res["fecha_ejecucion"]=EJECUCION
res.to_sql("dwh_kpi_resultados",engine,if_exists="append",index=False)
print("KPIs almacenados:\n"); display(res[["codigo","valor","detalle"]])

---
## Serialización del pipeline ETL (Día 11, Álvaro López)

Conforme a las prácticas de industrialización del máster, el pipeline de preprocesamiento
se serializa con `joblib` para que pueda ser reutilizado en el Objetivo 2 (modelado) sin
repetir las transformaciones. Se guarda también un `metadata_etl.json` con la información
de la ejecución.

In [ ]:
import joblib
from sklearn.preprocessing import LabelEncoder, OneHotEncoder
from sklearn.pipeline import Pipeline

# Pipeline de preprocesamiento reutilizable para datasets estructurados
# (las features numéricas de B1_MHSURVEY como ejemplo representativo)
d_mhs = pd.read_csv(os.path.join(BASE,"Bloque 1","1 Student Mental Health Survey","MentalHealthSurvey.csv"))
d_mhs.columns = [c.strip() for c in d_mhs.columns]
num_features = ["academic_workload","academic_pressure","financial_concerns",
                "social_relationships","isolation","future_insecurity",
                "study_satisfaction"]
X_mhs = d_mhs[num_features].fillna(d_mhs[num_features].median())
y_mhs = (d_mhs["depression"] >= 4).astype(int)  # binario: depresion alta

from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_mhs)

# Guardar pipeline con joblib (Día 11)
pipeline_path = os.path.join(ALMC, "pipeline_preprocesamiento_mhsurvey.pkl")
joblib.dump(scaler, pipeline_path)

# Guardar metadata
metadata = {
    "pipeline": "StandardScaler sobre MentalHealthSurvey",
    "features": num_features,
    "target": "depresion_alta (>=4)",
    "n_train": len(X_mhs),
    "fecha_ejecucion": EJECUCION.strftime("%Y-%m-%d %H:%M:%S"),
    "version": "1.0",
    "kpis": {r["codigo"]: r["valor"] for _, r in res.iterrows()}
}
meta_path = os.path.join(ALMC, "metadata_etl.json")
with open(meta_path, "w", encoding="utf-8") as f:
    json.dump(metadata, f, indent=2, ensure_ascii=False)

print(f"Pipeline serializado: {pipeline_path}")
print(f"Metadata guardada:    {meta_path}")
print(f"Cargar en Objetivo 2: scaler = joblib.load(r'{pipeline_path}')")

---
## Verificación final

Lee los KPIs desde la base uniendo catálogo + último resultado.
Esta consulta es la que alimentará las tarjetas de KPI en Power BI (Objetivo 3).

In [ ]:
q = """
SELECT c.codigo, c.nombre, r.valor, c.unidad, r.detalle, r.fecha_ejecucion
FROM dwh_kpi_resultados r
JOIN dwh_kpi_catalogo c ON c.id_kpi = r.id_kpi
WHERE r.fecha_ejecucion = (SELECT MAX(fecha_ejecucion) FROM dwh_kpi_resultados)
ORDER BY c.id_kpi
"""
display(pd.read_sql(text(q), engine))
print("\n✅ ETL ENTREGA 2 COMPLETADO — Notebook 08 v2")
print("   Tablas generadas: dwh_hechos_estres_unificado, dwh_agg_prevalencia,")
print("   dwh_nlp_senales, dwh_dass_severidad, dwh_brecha_tratamiento,")
print("   dwh_kpi_catalogo, dwh_kpi_resultados")
print(f"   Pipeline serializado en: {pipeline_path}")